# HỆ THỐNG PHÂN LOẠI BÌNH LUẬN ĐỘC HẠI
## Toxic Comment Classification Pipeline

---

### Mục Đích
Phân loại tự động các bình luận trên nền tảng truyền thông xã hội thành 6 danh mục:
- **toxic**: Bình luận độc hại
- **severe_toxic**: Bình luận độc hại cực độ
- **obscene**: Bình luận bỏng
- **threat**: Bình luận đe dọa
- **insult**: Bình luận xúc phạm
- **identity_hate**: Bình luận thù ghét danh tính

### Dữ Liệu
- **Nguồn:** Kaggle Toxic Comment Classification Dataset
- **Số lượng:** ~160,000 bình luận từ Wikipedia
- **Loại:** Multi-label classification (một bình luận có thể có 0, 1, hoặc nhiều nhãn)

### Pipeline
1. **Preprocessing**: Làm sạch văn bản
2. **Data Loading**: Chia dữ liệu train/test
3. **Feature Extraction**: TF-IDF Vectorization
4. **Model Training**: OneVsRest + Logistic Regression
5. **Evaluation**: Metrics và Visualizations
6. **Error Analysis**: Phân tích False Negatives và False Positives

## 1. IMPORT LIBRARIES

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# Metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    hamming_loss,
    ConfusionMatrixDisplay,
    multilabel_confusion_matrix,
    average_precision_score,
    auc,
    precision_recall_curve
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

## 2. DATA PREPROCESSING (XỬ LÝ DỮ LIỆU)

In [ ]:
# Load raw data
df = pd.read_csv("data/raw/train.csv")

print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Names:\n{df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Define labels
labels = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

print(f"\nNumber of Labels: {len(labels)}")
print(f"Labels: {labels}")

# Display label statistics
print(f"\n--- LABEL STATISTICS ---")
for label in labels:
    count = df[label].sum()
    percentage = (count / len(df)) * 100
    print(f"{label:15s}: {count:6d} ({percentage:5.2f}%)")

In [ ]:
# Function to clean text
def clean_text(text):
    """
    Clean text by:
    1. Convert to lowercase
    2. Remove special characters (keep only letters and spaces)
    3. Normalize whitespace
    """
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

# Apply cleaning
print("Cleaning text... This may take a moment...")
df["clean_text"] = df["comment_text"].apply(clean_text)

print("✓ Text cleaning completed!")
print(f"\nBefore cleaning:")
print(df["comment_text"].iloc[0][:100])
print(f"\nAfter cleaning:")
print(df["clean_text"].iloc[0][:100])

In [ ]:
# Drop original comment column and save processed data
df = df.drop(columns=["comment_text"])

# Create processed directory if not exists
import os
os.makedirs("data/processed", exist_ok=True)

# Save processed data
df.to_csv("data/processed/clean_train.csv", index=False)

print("✓ Processed data saved to 'data/processed/clean_train.csv'")
print(f"\nProcessed Dataset Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

## 3. DATA LOADING & SPLITTING (TẢI VÀ CHIA DỮ LIỆU)

In [ ]:
# Load cleaned data
df = pd.read_csv("data/processed/clean_train.csv")

# Fill missing values
df["clean_text"] = df["clean_text"].fillna("")
df["clean_text"] = df["clean_text"].astype(str)

# Prepare data for multi-label classification
X = df["clean_text"]
y = df[["toxic", "severe_toxic", "obscene", "insult", "threat", "identity_hate"]]

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"✓ Data loaded and split successfully!")
print(f"\nTrain Set:")
print(f"  - X_train shape: {X_train.shape}")
print(f"  - y_train shape: {y_train.shape}")
print(f"\nTest Set:")
print(f"  - X_test shape: {X_test.shape}")
print(f"  - y_test shape: {y_test.shape}")

In [ ]:
# Store original test data for error analysis
X_test_raw = X_test.copy()

print(f"Sample from training set:")
print(f"\nComment: {X_train.iloc[0]}")
print(f"\nLabels:")
print(y_train.iloc[0])

## 4. FEATURE EXTRACTION (TF-IDF VECTORIZATION)

In [ ]:
# TF-IDF Vectorization
print("Applying TF-IDF Vectorization...")

tfidf = TfidfVectorizer(max_features=5000)

# Fit on training data and transform both train and test
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"✓ TF-IDF Vectorization completed!")
print(f"\nVectorizer Configuration:")
print(f"  - Max Features: 5000")
print(f"  - Total Features Extracted: {len(tfidf.get_feature_names_out())}")
print(f"\nTransformed Data Shape:")
print(f"  - X_train_tfidf: {X_train_tfidf.shape}")
print(f"  - X_test_tfidf: {X_test_tfidf.shape}")
print(f"\nSparse Matrix (memory efficient):")
print(f"  - Train sparsity: {(1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])) * 100:.2f}%")
print(f"  - Test sparsity: {(1 - X_test_tfidf.nnz / (X_test_tfidf.shape[0] * X_test_tfidf.shape[1])) * 100:.2f}%")

In [ ]:
# Get top features (important words)
feature_names = np.array(tfidf.get_feature_names_out())

# Get mean TF-IDF scores for each feature
tfidf_scores = np.asarray(X_train_tfidf.mean(axis=0)).ravel()

# Get top 20 features
top_indices = tfidf_scores.argsort()[-20:][::-1]
top_features = feature_names[top_indices]
top_scores = tfidf_scores[top_indices]

print("Top 20 Important Features (Words):")
print("-" * 40)
for i, (feature, score) in enumerate(zip(top_features, top_scores), 1):
    print(f"{i:2d}. {feature:20s} - Score: {score:.4f}")

In [ ]:
# Visualize top features
plt.figure(figsize=(12, 6))
plt.barh(range(len(top_features)), top_scores, color='steelblue')
plt.yticks(range(len(top_features)), top_features)
plt.xlabel('Mean TF-IDF Score', fontsize=12)
plt.ylabel('Features (Words)', fontsize=12)
plt.title('Top 20 Important Words (Features) by TF-IDF Score', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('results/top_features_tfidf.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to 'results/top_features_tfidf.png'")

## 5. MODEL TRAINING (HUẤN LUYỆN MÔ HÌNH)

In [ ]:
# Train OneVsRest Multi-label Classification Model
print("Training OneVsRest Logistic Regression Model...\n")

label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Convert labels to numpy array
y_train_ml = y_train.values
y_test_ml = y_test.values

# Train model
model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000)
)

model.fit(X_train_tfidf, y_train_ml)

print("✓ Model training completed!")
print(f"\nModel Details:")
print(f"  - Model Type: OneVsRest Classifier")
print(f"  - Base Estimator: Logistic Regression")
print(f"  - Number of Sub-models: {len(model.estimators_)}")
print(f"  - Labels: {label_cols}")

In [ ]:
# Make predictions
print("Making predictions...\n")

y_pred = model.predict(X_test_tfidf)
y_prob = model.predict_proba(X_test_tfidf)

print("✓ Predictions completed!")
print(f"\nPrediction Shapes:")
print(f"  - y_pred shape: {y_pred.shape}")
print(f"  - y_prob shape: {y_prob.shape}")
print(f"\nSample Predictions:")
print(f"  First 3 samples:")
for i in range(3):
    print(f"\n  Sample {i+1}:")
    print(f"    Predicted Labels: {y_pred[i]}")
    print(f"    Probabilities: {[f'{p:.3f}' for p in y_prob[i]]}")

## 6. MODEL EVALUATION (ĐÁNH GIÁ MÔ HÌNH)

In [ ]:
# Classification Report
print("=" * 80)
print("CLASSIFICATION REPORT (Multi-Label Classification)")
print("=" * 80)
print(classification_report(y_test_ml, y_pred, target_names=label_cols, zero_division=0))

In [ ]:
# Hamming Loss
hamming = hamming_loss(y_test_ml, y_pred)
print("\n" + "=" * 80)
print("HAMMING LOSS (Fraction of incorrect labels)")
print("=" * 80)
print(f"Hamming Loss: {hamming:.4f}")
print(f"Interpretation: {hamming*100:.2f}% of labels are predicted incorrectly")
print(f"(Lower is better, 0 = perfect predictions)")

In [ ]:
# ROC-AUC and PR-AUC Scores
print("\n" + "=" * 80)
print("ROC-AUC & PR-AUC SCORES (Macro Average)")
print("=" * 80)

roc_auc_macro = roc_auc_score(y_test_ml, y_prob, average='macro')
pr_auc_macro = average_precision_score(y_test_ml, y_prob, average='macro')

print(f"ROC-AUC (macro): {roc_auc_macro:.4f}")
print(f"PR-AUC (macro):  {pr_auc_macro:.4f}")
print(f"\nInterpretation: (1.0 = perfect, 0.5 = random)")

# Per-label scores
print(f"\nPer-Label ROC-AUC Scores:")
print("-" * 40)
for i, label in enumerate(label_cols):
    roc_auc = roc_auc_score(y_test_ml[:, i], y_prob[:, i])
    print(f"{label:15s}: {roc_auc:.4f}")

In [ ]:
# Visualize ROC and PR Curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curves for each label
for i, label in enumerate(label_cols):
    fpr, tpr, _ = roc_curve(y_test_ml[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f"{label} (AUC={roc_auc:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
axes[0].set_xlabel("False Positive Rate", fontsize=12)
axes[0].set_ylabel("True Positive Rate", fontsize=12)
axes[0].set_title("ROC Curves - Multi-Label Classification", fontsize=14, fontweight='bold')
axes[0].legend(loc="lower right", fontsize=9)
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curves for each label
for i, label in enumerate(label_cols):
    precision, recall, _ = precision_recall_curve(y_test_ml[:, i], y_prob[:, i])
    pr_auc = auc(recall, precision)
    axes[1].plot(recall, precision, label=f"{label} (AUC={pr_auc:.3f})")

axes[1].set_xlabel("Recall", fontsize=12)
axes[1].set_ylabel("Precision", fontsize=12)
axes[1].set_title("Precision-Recall Curves - Multi-Label Classification", fontsize=14, fontweight='bold')
axes[1].legend(loc="upper right", fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/multilabel_roc_pr_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ ROC & PR Curves saved to 'results/multilabel_roc_pr_curves.png'")

In [ ]:
# Confusion Matrices for each label
mcm = multilabel_confusion_matrix(y_test_ml, y_pred)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Confusion Matrices - Multi-Label Classification', fontsize=16, fontweight='bold', y=1.00)

for i, (ax, label) in enumerate(zip(axes.flat, label_cols)):
    ConfusionMatrixDisplay(mcm[i], display_labels=['Not ' + label, label]).plot(
        ax=ax, cmap='Blues', values_format='d'
    )
    ax.set_title(f"Confusion Matrix - {label}", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('results/multilabel_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion Matrices saved to 'results/multilabel_confusion_matrices.png'")

## 7. ERROR ANALYSIS (PHÂN TÍCH LỖI)

In [ ]:
# Function to analyze errors
def analyze_errors(X_test_raw, y_test, y_pred, label_name, label_cols):
    """
    Analyze False Negatives and False Positives for a specific label
    """
    label_index = label_cols.index(label_name)
    
    # Create error dataframe
    df_errors = pd.DataFrame({
        'comment': list(X_test_raw),
        'true_label': list(y_test[:, label_index]),
        'predicted': list(y_pred[:, label_index])
    })
    
    # False Negatives: predicted 0 but actual is 1 (missed toxic comments)
    fn = df_errors[(df_errors['true_label'] == 1) & (df_errors['predicted'] == 0)]
    
    # False Positives: predicted 1 but actual is 0 (false alarms)
    fp = df_errors[(df_errors['true_label'] == 0) & (df_errors['predicted'] == 1)]
    
    return fn, fp

# Analyze errors for each label
print("\n" + "="*80)
print("ERROR ANALYSIS - FALSE NEGATIVES & FALSE POSITIVES")
print("="*80)

for label in label_cols:
    fn, fp = analyze_errors(X_test_raw, y_test_ml, y_pred, label, label_cols)
    
    print(f"\n{'-'*80}")
    print(f"LABEL: {label.upper()}")
    print(f"{'-'*80}")
    print(f"False Negatives (Missed - bỏ sót): {len(fn)} cases")
    print(f"False Positives (False Alarms - báo nhầm): {len(fp)} cases")
    
    # Show false negatives examples
    if len(fn) > 0:
        print(f"\nFalse Negatives Examples (First 3):")
        for idx, (i, row) in enumerate(fn.head(3).iterrows(), 1):
            print(f"\n  {idx}. {row['comment'][:100]}...")
    else:
        print(f"\nNo False Negatives found! ✓")
    
    # Show false positives examples
    if len(fp) > 0:
        print(f"\nFalse Positives Examples (First 3):")
        for idx, (i, row) in enumerate(fp.head(3).iterrows(), 1):
            print(f"\n  {idx}. {row['comment'][:100]}...")
    else:
        print(f"\nNo False Positives found! ✓")

## 8. SUMMARY & RECOMMENDATIONS (TÓM TẮT & ĐỀ XUẤT)

In [ ]:
### Key Achievements ✓

1. **Data Preprocessing Pipeline**
   - Loaded ~160,000 toxic comments from Kaggle
   - Applied text cleaning (lowercasing, special character removal, whitespace normalization)
   - Reduced data dimensionality while preserving semantic information

2. **Feature Extraction**
   - Implemented TF-IDF vectorization with 5,000 features
   - Identified top important words (features) for toxic comments
   - Created sparse matrices for efficient computation

3. **Multi-Label Classification Model**
   - Trained OneVsRest Logistic Regression for 6-label classification
   - Each label gets its own independent binary classifier
   - Allows flexible multi-label predictions

4. **Comprehensive Evaluation**
   - Classification reports with Precision, Recall, F1-Score
   - Hamming Loss for overall multi-label accuracy
   - ROC-AUC and PR-AUC scores for each label
   - Visualization of ROC curves, PR curves, and confusion matrices

5. **Error Analysis**
   - Identified False Negatives (missed toxic comments)
   - Identified False Positives (false alarms)
   - Examples for each label provided for manual review

---

### Recommendations for Improvement

| Aspect | Current | Recommendation |
|--------|---------|----------------|
| **Text Processing** | Basic cleaning | Add stemming/lemmatization, handle contractions |
| **Feature Extraction** | TF-IDF | Try Word2Vec, GloVe, or BERT embeddings |
| **Model Architecture** | Linear models | Try Random Forest, Gradient Boosting, Neural Networks |
| **Class Imbalance** | class_weight='balanced' | Try SMOTE, weighted loss functions |
| **Threshold** | Default 0.5 | Optimize per-label threshold based on F1-score |
| **Deep Learning** | Not used | Implement LSTM, CNN, or Transformer models |

---

### Output Files Generated

- ✅ `data/processed/clean_train.csv` - Cleaned data
- ✅ `results/top_features_tfidf.png` - Top 20 important words
- ✅ `results/multilabel_roc_pr_curves.png` - ROC & PR curves
- ✅ `results/multilabel_confusion_matrices.png` - 6 confusion matrices

In [ ]:
# Final Summary Statistics
print("\n" + "="*80)
print("FINAL SUMMARY STATISTICS")
print("="*80)

print(f"\n📊 Dataset:")
print(f"  - Total samples: {len(df)}")
print(f"  - Train samples: {len(X_train)}")
print(f"  - Test samples: {len(X_test)}")

print(f"\n🔤 Text Processing:")
print(f"  - TF-IDF features: 5,000")
print(f"  - Train sparsity: {(1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])) * 100:.2f}%")

print(f"\n🤖 Model:")
print(f"  - Type: OneVsRest Classifier")
print(f"  - Base: Logistic Regression")
print(f"  - Labels: 6")

print(f"\n📈 Performance:")
print(f"  - Hamming Loss: {hamming:.4f}")
print(f"  - ROC-AUC (macro): {roc_auc_macro:.4f}")
print(f"  - PR-AUC (macro): {pr_auc_macro:.4f}")

print(f"\n✓ Pipeline completed successfully!")